# Understanding error messages

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 6**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* say what an error message is and why it is your friend
* read a **traceback** from the bottom up
* tell **syntax errors**, **runtime errors** and **logic errors** apart
* recognise the common error types: `SyntaxError`, `IndentationError`, `NameError`, `TypeError`, `ValueError`, `ZeroDivisionError`, `IndexError`, `KeyError`, `AttributeError`, `FileNotFoundError`
* catch an error with `try` / `except` so your program does not crash
* follow a short debugging checklist

## 2. Errors are information

Every programmer sees errors every day. An error message is Python telling you **what went wrong** and **where**.
Do not panic and do not skip it. Read it.

A tiny helper first. The cell below defines `run(code)`. It runs a piece of code (given as text) and, if the code fails,
prints the error message the way Python would. This lets us study many errors in one notebook without the notebook stopping.

In [1]:
import traceback, linecache

def run(source, name="<lesson>"):
    '''Run a piece of code. If it fails, print the traceback Python would show.'''
    linecache.cache[name] = (len(source), None, source.splitlines(True), name)   # lets the traceback show our lines
    try:
        code = compile(source, name, "exec")
    except SyntaxError as e:                       # the code could not even be read
        print("".join(traceback.format_exception(type(e), e, None)))
        return
    try:
        exec(code, {})
    except Exception as e:                         # the code was read, but failed while running
        print("".join(traceback.format_exception(type(e), e, e.__traceback__.tb_next)))

run("print('this works')")

this works


## 3. Three kinds of errors

| Kind | When it shows up | Does Python tell you? |
|---|---|---|
| **Syntax error** | before the program starts, while Python reads the code | yes, always |
| **Runtime error** (exception) | while the program runs | yes, with a traceback |
| **Logic error** | the program runs and finishes | **no**. The answer is simply wrong |

Logic errors are the hardest, because nothing crashes. Here is one. We want the **average** of 10, 20 and 30.

In [2]:
a, b, c = 10, 20, 30
average = a + b + c / 3          # wrong: division happens before the additions
print(average)                   # prints 40.0, but the true average is 20.0

average = (a + b + c) / 3        # fixed with brackets
print(average)

40.0
20.0


Python printed a number and no error. Only **you** know that 40 is wrong. Always check your results with a small example you can do by hand.

## 4. Reading a traceback

When a runtime error is not handled, Python prints a **traceback**. Here is a small program with a bug in a function that calls another function.

In [3]:
program = '''
def average(numbers):
    return sum(numbers) / len(numbers)

def report(marks):
    print("Average is", average(marks))

report([])
'''
run(program)

Traceback (most recent call last):
  File "<lesson>", line 8, in <module>
    report([])
    ~~~~~~^^^^
  File "<lesson>", line 6, in report
    print("Average is", average(marks))
                        ~~~~~~~^^^^^^^
  File "<lesson>", line 3, in average
    return sum(numbers) / len(numbers)
           ~~~~~~~~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero



### Read it from the bottom up

1. **Last line** (the most important): `ZeroDivisionError: division by zero`. It gives the **error type** and a **message**.
2. **The line just above**: the exact line that failed, `return sum(numbers) / len(numbers)`, and the file and line number.
3. **Lines further up**: the chain of function calls that led there. `report` called `average`, and `<module>` (the main program) called `report`.

So the story is: the main program called `report([])`, which called `average([])`, and `len([])` is 0, so dividing by it failed.
The `^^^^` marks show the part of the line that failed.

The bug is not really on the last line. The real cause is that we passed an **empty list**. The traceback shows where to start looking.

## 5. Syntax errors

A syntax error means Python cannot understand the code, like a sentence with broken grammar.
Nothing runs at all, not even the lines before the mistake.
Typical causes: a missing colon, a missing bracket or quote, a wrong keyword.

In [4]:
run("if 5 > 3\n    print('yes')")          # missing colon

  File "<lesson>", line 1
    if 5 > 3
            ^
SyntaxError: expected ':'



In [5]:
run("print('hello'")                       # missing closing bracket

  File "<lesson>", line 1
    print('hello'
         ^
SyntaxError: '(' was never closed



In [6]:
run("name = 'Asha\nprint(name)")           # missing closing quote

  File "<lesson>", line 1
    name = 'Asha
           ^
SyntaxError: unterminated string literal (detected at line 1)



In [7]:
run("x = 5\nif x = 5:\n    print('five')")    # = used instead of ==

  File "<lesson>", line 2
    if x = 5:
       ^^^^^
SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='?



Python often points at the line **after** the real mistake, because it only notices the problem when it gets confused.
If the marked line looks fine, check the line above it.

### IndentationError
`IndentationError` is a special kind of `SyntaxError` about the spaces at the start of a line.

In [8]:
run("for i in range(3):\nprint(i)")                      # block not indented

  File "<lesson>", line 2
    print(i)
    ^^^^^
IndentationError: expected an indented block after 'for' statement on line 1



In [9]:
run("x = 1\n    y = 2")                                   # indented for no reason

  File "<lesson>", line 2
    y = 2
IndentationError: unexpected indent



## 6. Runtime errors (exceptions)

Here the code is valid, but something goes wrong while it runs. Python **raises an exception**. Let us meet the common ones.

### NameError
You used a name that does not exist. Usually a typo, or you forgot to create the variable first.

In [10]:
run("total = 10\nprint(totl)")

Traceback (most recent call last):
  File "<lesson>", line 2, in <module>
    print(totl)
          ^^^^
NameError: name 'totl' is not defined. Did you mean: 'total'?



### TypeError
You used a value of the wrong type for an operation, for example adding a number to text, or calling something that is not a function.

In [11]:
run("print('Age: ' + 20)")

Traceback (most recent call last):
  File "<lesson>", line 1, in <module>
    print('Age: ' + 20)
          ~~~~~~~~^~~~
TypeError: can only concatenate str (not "int") to str



In [12]:
run("print(len(5))")

Traceback (most recent call last):
  File "<lesson>", line 1, in <module>
    print(len(5))
          ~~~^^^
TypeError: object of type 'int' has no len()



In [13]:
run("x = 10\nx()")

Traceback (most recent call last):
  File "<lesson>", line 2, in <module>
    x()
    ~^^
TypeError: 'int' object is not callable



### ValueError
The type is right, but the **value** is not acceptable.

In [14]:
run("n = int('twelve')")

Traceback (most recent call last):
  File "<lesson>", line 1, in <module>
    n = int('twelve')
ValueError: invalid literal for int() with base 10: 'twelve'



### ZeroDivisionError
Dividing by zero (`/`, `//` and `%` all do it).

In [15]:
run("print(10 / 0)")

Traceback (most recent call last):
  File "<lesson>", line 1, in <module>
    print(10 / 0)
          ~~~^~~
ZeroDivisionError: division by zero



### IndexError
You asked for a position that is not in a list or string. A list with 3 items has positions 0, 1 and 2.

In [16]:
run("colours = ['red', 'green', 'blue']\nprint(colours[3])")

Traceback (most recent call last):
  File "<lesson>", line 2, in <module>
    print(colours[3])
          ~~~~~~~^^^
IndexError: list index out of range



### KeyError
You asked a dictionary for a key it does not have.

In [17]:
run("marks = {'Asha': 80, 'Ravi': 65}\nprint(marks['Meena'])")

Traceback (most recent call last):
  File "<lesson>", line 2, in <module>
    print(marks['Meena'])
          ~~~~~^^^^^^^^^
KeyError: 'Meena'



### AttributeError
You asked an object for a method or property it does not have. Often a typo, or the wrong type of object.

In [18]:
run("name = 'asha'\nprint(name.uppercase())")

Traceback (most recent call last):
  File "<lesson>", line 2, in <module>
    print(name.uppercase())
          ^^^^^^^^^^^^^^
AttributeError: 'str' object has no attribute 'uppercase'



### FileNotFoundError
You tried to open a file that is not there. Python raises this when the name or folder is wrong.

In [19]:
run("f = open('no_such_file.txt')")

Traceback (most recent call last):
  File "<lesson>", line 1, in <module>
    f = open('no_such_file.txt')
FileNotFoundError: [Errno 2] No such file or directory: 'no_such_file.txt'



## 7. Handling errors with `try` and `except`

Sometimes an error is expected, for example when a user types text where you wanted a number.
Instead of crashing, you can **catch** the exception.

```
try:
    risky code
except SomeError:
    what to do if it fails
```

Python runs the `try` block. If an error of that type happens, it jumps to the `except` block. If not, `except` is skipped.

In [20]:
text = "twelve"
try:
    number = int(text)
    print("You typed", number)
except ValueError:
    print("That is not a whole number:", text)

print("The program carries on.")

That is not a whole number: twelve
The program carries on.


You can catch several types, and get the error object with `as e`. `else` runs when nothing went wrong. `finally` always runs.

In [21]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  cannot divide by zero")
        return None
    except TypeError as e:
        print("  bad types:", e)
        return None
    else:
        print("  no error")
        return result
    finally:
        print("  finished", a, "/", b)

print(safe_divide(10, 4))
print(safe_divide(10, 0))
print(safe_divide(10, "2"))

  no error
  finished 10 / 4
2.5
  cannot divide by zero
  finished 10 / 0
None
  bad types: unsupported operand type(s) for /: 'int' and 'str'
  finished 10 / 2
None


You can also **raise** an error yourself when something is not acceptable.

In [22]:
def set_age(age):
    if age < 0:
        raise ValueError("age cannot be negative")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("Caught:", e)

Caught: age cannot be negative


## 8. Edge cases

**1. A bare `except:` hides every bug.** It even hides typos. Always name the error type you expect.

In [23]:
total = 0
try:
    total = totl + 1          # typo: NameError
except:                       # bad habit: catches everything
    print("something went wrong")      # no clue what
print(total)

try:
    total = totl + 1
except ZeroDivisionError:
    pass
except NameError as e:        # better: precise and shows the message
    print("NameError:", e)

something went wrong
0
NameError: name 'totl' is not defined


**2. Order of `except` blocks matters.** Python uses the **first** block that matches. `LookupError` is the parent of both `IndexError` and `KeyError`, and `Exception` is the parent of nearly everything. Put specific types first.

In [24]:
print(IndexError.__mro__)          # the family tree, child to parent

try:
    [1, 2, 3][10]
except LookupError:                # parent type also catches IndexError
    print("caught by LookupError")

(<class 'IndexError'>, <class 'LookupError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
caught by LookupError


**3. Code after the error never runs.** Inside a `try`, the lines after the failing line are skipped.

In [25]:
try:
    print("step 1")
    print(1 / 0)
    print("step 3")            # never reached
except ZeroDivisionError:
    print("stopped at step 2")

step 1
stopped at step 2


**4. Same error type, different cause.** Always read the **message**, not only the type.

In [26]:
for source in ["int('abc')", "int(None)", "int([1])", "'abc'.index('z')", "float('1,5')"]:
    try:
        eval(source)
    except Exception as e:
        print(f"{source:18} -> {type(e).__name__}: {e}")

int('abc')         -> ValueError: invalid literal for int() with base 10: 'abc'
int(None)          -> TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
int([1])           -> TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'
'abc'.index('z')   -> ValueError: substring not found
float('1,5')       -> ValueError: could not convert string to float: '1,5'


Notice that `int('abc')` is a `ValueError` (right type, bad value) but `int(None)` is a `TypeError` (wrong type).

**5. `input()` always gives text.** `int(input())` can raise `ValueError`. Test your code with text, an empty string and a decimal such as `"3.5"`.

In [27]:
for typed in ["42", "", "3.5", " 7 ", "-2"]:
    try:
        print(repr(typed), "->", int(typed))
    except ValueError as e:
        print(repr(typed), "-> ValueError:", e)

'42' -> 42
'' -> ValueError: invalid literal for int() with base 10: ''
'3.5' -> ValueError: invalid literal for int() with base 10: '3.5'
' 7 ' -> 7
'-2' -> -2


## 9. A debugging checklist

1. **Read the last line** of the traceback. Note the error type and message.
2. **Find the line number** and look at that line **and the one above it**.
3. **Check the names and types** on that line. `print(type(x))` and `print(x)` often solve it.
4. **Run a smaller piece.** Test one line or one function with simple input.
5. **Print values** at several points to see where reality leaves your plan.
6. **Compare with a hand calculation** for logic errors.
7. **Change one thing at a time**, then run again.
8. **Explain the code out loud** to a friend (or a rubber duck). You often spot the bug while talking.
9. **Search the exact error message** if you are still stuck.

## 10. Quick summary

| Error | Typical cause | Example |
|---|---|---|
| `SyntaxError` | broken grammar: missing `:`, bracket, quote | `if x > 3` |
| `IndentationError` | wrong spaces at the start of a line | block not indented |
| `NameError` | name not defined (typo, or used too early) | `print(totl)` |
| `TypeError` | wrong type for an operation | `"a" + 1` |
| `ValueError` | right type, unacceptable value | `int("abc")` |
| `ZeroDivisionError` | dividing by zero | `5 / 0` |
| `IndexError` | position outside a list or string | `[1, 2][5]` |
| `KeyError` | missing dictionary key | `d["missing"]` |
| `AttributeError` | object has no such method or property | `"a".uppercase()` |
| `FileNotFoundError` | file or folder does not exist | `open("none.txt")` |

* Read a traceback **bottom-up**: type and message, then the line, then the call chain.
* Syntax errors stop the program before it starts. Runtime errors stop it midway. Logic errors do not stop it at all.
* Catch **specific** errors with `try` / `except`. Never hide bugs with a bare `except`.

## 11. Exercises

### Exercise 1

Name the error type (without running the code) for each line. Then check by running each inside `try` / `except Exception as e`, printing `type(e).__name__`.

1. `print(undefined_name)`
2. `int("4.5")`
3. `[1, 2, 3][3]`
4. `{"a": 1}["b"]`
5. `"abc" * "2"`
6. `5 // 0`

*Hint:* use `eval(...)` on each text, inside a `try` block.

In [28]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

Answers: NameError, ValueError, IndexError, KeyError, TypeError, ZeroDivisionError. `int('4.5')` fails because a whole-number parser does not accept a decimal point.

```python
tests = ["undefined_name", "int('4.5')", "[1, 2, 3][3]", "{'a': 1}['b']", "'abc' * '2'", "5 // 0"]
for t in tests:
    try:
        eval(t)
    except Exception as e:
        print(f"{t:18} {type(e).__name__}")
```

</details>

### Exercise 2

This code has a syntax error. Find all the problems, fix them, and make it print `Pass`.

```python
marks = 55
if marks >= 40
    print("Pass"
else:
    print("Fail")
```

*Hint:* two small things are missing.

In [29]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

The `if` line needed a colon, and the `print(` call needed its closing bracket.

```python
marks = 55
if marks >= 40:
    print("Pass")
else:
    print("Fail")
```

</details>

### Exercise 3

Write a function `to_int(text)` that returns `int(text)` if possible, and returns `-1` if the text is not a whole number. Test it with `"25"`, `"abc"`, `""` and `"3.7"`.

In [30]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

Output: 25, -1, -1, -1. We catch only `ValueError`, so other bugs would still be visible.

```python
def to_int(text):
    try:
        return int(text)
    except ValueError:
        return -1

for t in ["25", "abc", "", "3.7"]:
    print(repr(t), to_int(t))
```

</details>

### Exercise 4

Make the traceback readable. The code below fails. Run it inside `try` / `except`, and print only the error type, the message, and the **line number** of the failing line (use `e.__traceback__.tb_lineno`).

```python
data = [4, 8, 15]
index = 5
print(data[index])
```

*Hint:* inside `except IndexError as e:` use `type(e).__name__`, `e` and `e.__traceback__.tb_lineno`. The line number depends on the cell, so do not worry about its exact value.

In [31]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

The message `list index out of range` says the position is too big. A list of 3 items only has positions 0, 1, 2.

```python
data = [4, 8, 15]
index = 5
try:
    print(data[index])
except IndexError as e:
    print("Type:", type(e).__name__)
    print("Message:", e)
    print("Line number:", e.__traceback__.tb_lineno)
```

</details>

### Exercise 5

Logic error hunt. This code should print the total price including 18 percent tax for a price of 500, which is 590. Run it, see the wrong output, fix it.

```python
price = 500
total = price + price * 18 / 100 * 100
print(total)
```

In [32]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

The extra `* 100` was wrong. The tax is `500 * 18 / 100 = 90`. Python gave no error because the original code was valid. Output: 590.0.

```python
price = 500
total = price + price * 18 / 100
print(total)
```

</details>

### Exercise 6

Write a function `get_mark(marks, name)` that takes a dictionary of marks. It returns the mark if the name is present, and otherwise returns the text `"no such student"`. Use `try` / `except KeyError`. Test with `{"Asha": 80, "Ravi": 65}` and the names `"Asha"` and `"Meena"`.

In [33]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

Output: 80 then no such student. (The method `marks.get(name, default)` does the same job in one line.)

```python
def get_mark(marks, name):
    try:
        return marks[name]
    except KeyError:
        return "no such student"

marks = {"Asha": 80, "Ravi": 65}
print(get_mark(marks, "Asha"))
print(get_mark(marks, "Meena"))
```

</details>

### Exercise 7

**Edge case.** Predict the output of this code, then run it. Why is the `finally` line printed even though the function returns inside `try`?

```python
def f():
    try:
        return "from try"
    finally:
        print("cleanup")

print(f())
```

In [34]:
# Write your answer here

<details>
<summary><b>Show solution 7</b></summary>

Output: `cleanup` then `from try`. A `finally` block always runs before the function really leaves, even after a `return`. That is why it is used for clean-up jobs such as closing files.

```python
def f():
    try:
        return "from try"
    finally:
        print("cleanup")

print(f())
```

</details>

## 12. Where to read more

* Software Carpentry, *Errors and Exceptions*: https://swcarpentry.github.io/python-novice-inflammation/09-errors.html
* Real Python, *Understanding the Python Traceback*: https://realpython.com/python-traceback/